# PEMS-SF 数据集初步探索 (EDA)
数据来源: https://archive.ics.uci.edu/dataset/204/pems+sf 
加州高速公路 PeMS 旧金山湾区交通传感器数据。目标: 对 data/ 目录中已有文件做结构与基础统计、缺失与异常初查、形成元数据表与后续建议。

## 任务概述
1. 元数据表: 文件, 记录数, 特征数, 数值范围, 缺失情况。
2. 观测数与特征数: 总体与(若能)按日。
3. 分组: 标签/站点类型/地理分布。
4. 缺失值与无效值检查。
5. 列出特征 (当前仅可解析的矩阵列占位)。
6. 后续处理建议。

## 数据文件初步说明
data/ 中包含: PEMS_train, PEMS_test (可能为主数值矩阵); PEMS_trainlabels, PEMS_testlabels (分类/站点标签); randperm (随机索引); stations_list (站点信息)。格式需读取验证。

In [ ]:
# 导入库
import pathlib, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
plt.style.use('seaborn-v0_8')
DATA_DIR = pathlib.Path('data')

In [ ]:
# 统一设置中文字体与负号显示，避免中文内容或负号无法显示
import matplotlib
from matplotlib import font_manager, rcParams
# 优先选择系统常见中文字体，按可用性顺序
rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'Noto Sans CJK SC']
rcParams['axes.unicode_minus'] = False
# 如需强制指定字体文件，可取消以下两行注释并设置路径为系统实际存在的中文字体
# font_path = r'C:\Windows\Fonts\msyh.ttc'  # 微软雅黑
# matplotlib.font_manager.fontManager.addfont(font_path); rcParams['font.sans-serif'] = ['Microsoft YaHei']
print('当前字体配置：', rcParams['font.sans-serif'], 'unicode_minus=', rcParams['axes.unicode_minus'])

# 新增：英文绘图文本映射与助手（放在最前面，注释中文）
plot_texts = {
    'train_label_dist_title': 'Train label distribution',
    'train_daily_mean_boxplot': 'Train: daily mean occupancy by weekday label',
    'avg_daily_profile_by_label': 'Average daily occupancy profile (by label)',
    'time_slot_xlabel': 'Time slot (10-min units, 144 slots)',
    'avg_occupancy_ylabel': 'Average occupancy',
    'station_weekday_profile_title': 'Station {sid}: Mean occupancy per time slot by weekday (1-7)',
    'station_daily_mean_title': 'Station {sid}: Daily mean occupancy by weekday label',
    'station_plot_xlabel': 'Weekday label (1=Mon ... 7=Sun)',
    'station_plot_ylabel': 'Daily mean occupancy',
}
# 中文注释：L 用于获取英文文本，支持 format 参数，如 L('station_weekday_profile_title', sid=123)
def L(key, **kw):
    s = plot_texts.get(key, key)
    return s.format(**kw)

In [ ]:
# 列出文件基础信息
file_info = []
for p in DATA_DIR.iterdir():
    if p.is_file():
        try: content = p.read_text(encoding='utf-8', errors='ignore')[:160].replace('\n',' ')
        except Exception: content = ''
        file_info.append({'filename': p.name, 'size_bytes': p.stat().st_size, 'preview': content})
pd.DataFrame(file_info)

In [ ]:
# 读取标签文件 (空白分隔, 去除方括号)
def load_label_file(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    nums = [int(x) for x in txt.split() if re.match(r'^-?\d+$', x)]
    return np.array(nums, dtype=int) if nums else None
labels_train = load_label_file(DATA_DIR/'PEMS_trainlabels') if (DATA_DIR/'PEMS_trainlabels').exists() else None
labels_test  = load_label_file(DATA_DIR/'PEMS_testlabels')  if (DATA_DIR/'PEMS_testlabels').exists()  else None
labels_summary = {}
for name, arr in [('train', labels_train), ('test', labels_test)]:
    if arr is not None:
        labels_summary[name] = {
            'count': int(arr.size),
            'unique': np.unique(arr).tolist(),
            'value_counts': dict(Counter(arr)),
            'min': int(arr.min()),
            'max': int(arr.max())
        }
labels_summary

### 标签分布解释
若 unique 为 1..7 代表 7 类; 需与 UCI 数据说明对照。检查是否类不平衡, 为后续建模准备采样策略。

In [ ]:
# 标签分布可视化
def plot_label_dist(arr, title):
    vc = pd.Series(arr).value_counts().sort_index()
    plt.figure(figsize=(6,3))
    sns.barplot(x=vc.index, y=vc.values, palette='tab10')
    plt.title(title); plt.xlabel('Label'); plt.ylabel('Count'); plt.tight_layout(); plt.show()
if labels_train is not None: plot_label_dist(labels_train, 'Train Label Distribution')
if labels_test is not None: plot_label_dist(labels_test, 'Test Label Distribution')

In [ ]:
# 尝试读取主矩阵 (兼容两种格式：
# 1) 每行是一个数值向量（所有行长度相同），
# 2) 每行是一个 Matlab 样式的日矩阵（含分号作为行分隔，例如 '[v1; v2; ...]')。
# 返回：当能解析为规则矩阵时返回 numpy 2D array；若行长度不一则用 NaN 填充返回矩阵以便后续检查。
def try_load_matrix(path: pathlib.Path, max_rows=200):
    if not path.exists(): return None
    txt = path.read_text(encoding='utf-8', errors='ignore')
    # 不先去掉中括号，以便识别 Matlab 风格的分号行分隔
    lines = [l for l in txt.split('\n') if l.strip()]
    # 优先检测 Matlab 风格的单行日矩阵（行内含 ';' 或方括号），把该行解析为 2D 数组并返回
    for i, l in enumerate(lines[:max_rows]):
        if (';' in l) or ('[' in l) or (']' in l):
            # 尝试按分号分割为多行，再解析每行的数值
            raw = l.strip()
            raw = raw.lstrip('[').rstrip(']')
            row_strs = [r.strip() for r in raw.split(';') if r.strip()]
            data = []
            for r in row_strs:
                nums = [float(x) for x in r.split() if re.match(r'^-?\d+(\.\d+)?$', x)]
                if nums: data.append(nums)
            if not data: return None
            # 尝试转换为规则矩阵；若列数不一致则返回不规则矩阵以便诊断
            lengths = [len(rr) for rr in data]
            if len(set(lengths)) == 1:
                return np.array(data, dtype=float)
            else:
                # 列数不一致，使用 NaN 填充至最大列数并返回
                maxc = max(lengths)
                mat = np.full((len(data), maxc), np.nan, dtype=float)
                for ii, rr in enumerate(data):
                    mat[ii, :len(rr)] = rr
                return mat
    # 否则把每行当作向量解析，收集最多 max_rows 行作为预览
    data = []
    for l in lines[:max_rows]:
        nums = [float(x) for x in l.split() if re.match(r'^-?\d+(\.\d+)?$', x)]
        if nums: data.append(nums)
    if not data: return None
    lengths = [len(r) for r in data]
    if len(set(lengths)) == 1:
        # 规则矩阵，直接返回堆叠结果
        return np.vstack(data)
    else:
        # 行长度不一致，填充 NaN 并返回，避免 ValueError
        maxc = max(lengths)
        mat = np.full((len(data), maxc), np.nan, dtype=float)
        for i, r in enumerate(data):
            mat[i, :len(r)] = r
        return mat
matrix_train = try_load_matrix(DATA_DIR/'PEMS_train')
matrix_test  = try_load_matrix(DATA_DIR/'PEMS_test')
{'train_shape_preview': None if matrix_train is None else matrix_train.shape, 'test_shape_preview': None if matrix_test is None else matrix_test.shape}

In [ ]:
# 数值统计 (预览矩阵)
summary_stats = {}
for name, mat in [('train', matrix_train), ('test', matrix_test)]:
    if mat is not None:
        summary_stats[name] = {
            'rows_preview': mat.shape[0],
            'cols': mat.shape[1] if mat.ndim>1 else 1,
            'min': float(np.nanmin(mat)),
            'max': float(np.nanmax(mat)),
            'mean': float(np.nanmean(mat)),
            'std': float(np.nanstd(mat)),
            'nan_count': int(np.isnan(mat).sum())
        }
summary_stats

In [ ]:
# 元数据表构建
meta_rows = []
for fname, arr in [('PEMS_trainlabels', labels_train), ('PEMS_testlabels', labels_test)]:
    if arr is not None:
        meta_rows.append({
            'file': fname,
            'n_observations': int(arr.size),
            'n_features': 1,
            'value_range': f'{arr.min()}..{arr.max()}',
            'missing': 0,
            'remark': '标签 (可能站点/类别)'
        })
for fname, mat in [('PEMS_train', matrix_train), ('PEMS_test', matrix_test)]:
    if mat is not None:
        meta_rows.append({
            'file': fname,
            'n_observations': mat.shape[0],
            'n_features': mat.shape[1] if mat.ndim>1 else 1,
            'value_range': f'{np.nanmin(mat)}..{np.nanmax(mat)}',
            'missing': int(np.isnan(mat).sum()),
            'remark': '数值矩阵 (时间x特征 需确认)'
        })
for other in ['randperm', 'stations_list']:
    p = DATA_DIR/other
    if p.exists(): meta_rows.append({'file': other, 'n_observations': None, 'n_features': None, 'value_range': None, 'missing': None, 'remark': '待解析'})
meta_df = pd.DataFrame(meta_rows)
meta_df

In [ ]:
# 占位: 按日聚合 (假设5分钟间隔 -> 288条/天) 若后续确认行=时间步
def group_by_day_placeholder(mat: np.ndarray, interval_per_day: int = 288):
    if mat is None: return None
    days = mat.shape[0] // interval_per_day
    if days == 0: return None
    daily = [np.nanmean(mat[i*interval_per_day:(i+1)*interval_per_day], axis=0) for i in range(days)]
    return np.vstack(daily)
daily_train = group_by_day_placeholder(matrix_train)
'daily_train_shape' if daily_train is not None else '无足够数据构成完整天'

## 缺失与异常值初查
1. 统计 NaN 数量。
2. 后续需检查物理不合理值 (如速度<0, 占有率>100%).
3. 低方差或常量列剔除。
4. 与 stations_list 结合提取站点分组。

## 下一步建议
1. 明确主数据格式 (确认是否时间序列: 行=时间步, 列=传感器或特征)。
2. 解析 stations_list 内容以获得地理/功能分组。
3. 若需要按日统计，需获得真实时间戳或默认采样间隔。
4. 检查标签语义 (7 类含义) 并与 UCI 说明对照。
5. 构建更完整的特征描述表 (单位、含义)。
6. 评估类别不平衡对建模的影响。
7. 规划数据清洗与建模管线 (缺失值插补、归一化、分割、评估指标)。

## 基于官方站点描述的补充说明
- 时间范围: 2008-01-01 到 2009-03-30 共 15 个月, 去除公共假期与两个异常日后共 440 天。
- 任务: 预测每天是星期几 (1=周一,...,7=周日)。
- 采样频率: 每 10 分钟一次, 一天 24*6=144 时间戳。
- 传感器/站点: 963 个稳定运行的检测器。
- 单日数据结构: 963 x 144 矩阵, 数值为占有率 (occupancy rate) 介于 0 与 1。
- 文件结构: PEMS_train 有 263 行; 每行是一个日矩阵的 Matlab 样式字符串, 例如 [row1; row2; ...]. PEMS_test 有 173 行; 总计 263+173=436 天 (官网声称 440, 可能已剔除更多异常或截图信息 有待核对)。
- 标签文件: 与对应行顺序对齐, 标签取值 1..7。
- randperm: 打乱顺序所用的排列; 还原原始日历序列需其逆排列。
- stations_list: 站点 ID 列表 (应为 963 条)。
- 官方说明: 无缺失值 (Has Missing Values? No)。我们仍做程序验证。

### 解析一行 Matlab 风格日矩阵 -> numpy 数组 (963x144)

In [ ]:
import math
def parse_day_matrix(line: str, expected_rows=963, expected_cols=144):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    # 按分号分行 (Matlab 行分隔)
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if re.match(r'^-?\d+(\.\d+)?$', x)]
        if nums: data.append(nums)
    if not data:
        return None
    # 处理行长度不一致的情况：使用 NaN 填充到最大列数，避免 np.array(..., dtype=float) 抛错
    lengths = [len(rr) for rr in data]
    max_len = max(lengths)
    if max_len == 0:
        return None
    if len(set(lengths)) == 1:
        # 行长度一致，直接转换为数组
        arr = np.array(data, dtype=float)
    else:
        # 行长度不一致，使用 NaN 填充
        arr = np.full((len(data), max_len), np.nan, dtype=float)
        for i, rr in enumerate(data):
            arr[i, :len(rr)] = rr
    # 仅在尺寸满足预期时返回, 否则返回实际形状以便诊断
    if arr.shape == (expected_rows, expected_cols):
        return arr
    return arr  # 返回实际形状以便诊断

### 流式迭代读取训练/测试文件 (避免一次性载入全部以节省内存)

In [ ]:
def iter_day_matrices_optimized(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            yield parse_day_matrix(line)

### 统计全局占有率分布 (Welford 在线算法) 与每日日统计 (min/max/mean) 仅预览前若干天

In [ ]:
def stream_global_optimized(path: pathlib.Path, sample_target=2_000_000):
    import random
    gmin, gmax = float('inf'), float('-inf')
    nan_ct = 0
    value_ct = 0
    mean = 0.0
    M2 = 0.0
    sample = []

    for mat in iter_day_matrices_optimized(path):
        vals = mat.ravel()
        day_min = vals.min()
        day_max = vals.max()
        gmin = min(gmin, day_min)
        gmax = max(gmax, day_max)
        nan_ct += np.isnan(vals).sum()

        for v in vals:
            if np.isnan(v):
                continue
            value_ct += 1
            delta = v - mean
            mean += delta / value_ct
            M2 += delta * (v - mean)

            if len(sample) < sample_target:
                sample.append(v)
            else:
                j = random.randint(0, value_ct - 1)
                if j < sample_target:
                    sample[j] = v

    variance = M2 / (value_ct - 1) if value_ct > 1 else 0.0
    sample_sorted = np.sort(sample)
    quantiles = {
        q: float(sample_sorted[int(q * (len(sample_sorted) - 1))])
        for q in [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1]
    }

    return {
        'global_min': float(gmin),
        'global_max': float(gmax),
        'nan_count': int(nan_ct),
        'value_count': int(value_ct),
        'mean': float(mean),
        'std': float(np.sqrt(variance)),
        'approx_quantiles': quantiles,
    }

In [ ]:
# 示例调用
train_path = pathlib.Path('data/PEMS_train')
test_path = pathlib.Path('data/PEMS_test')
train_stats_preview = stream_global_optimized(train_path, sample_target=1_000_000)
test_stats_preview = stream_global_optimized(test_path, sample_target=600_000)
train_stats_preview, test_stats_preview

### 占有率数值有效性检查 (是否全部在 [0,1] 内, 是否存在 NaN)

In [ ]:
def occupancy_validity(path: pathlib.Path, max_days=None):
    out_of_range = 0
    nan_count = 0
    day_total = 0
    for day_i, mat in enumerate(iter_day_matrices_optimized(path, limit=max_days)):
        day_total += 1
        vals = mat.ravel()
        nan_count += np.isnan(vals).sum()
        out_of_range += np.sum((vals < 0) | (vals > 1))
    return {'days_checked': day_total, 'nan_values': int(nan_count), 'out_of_range_values': int(out_of_range)}
validity_train_preview = occupancy_validity(train_path, max_days=10)
validity_train_preview

### 解析 randperm (还原日历顺序所需的排列)

In [ ]:
randperm_path = DATA_DIR/'randperm'
if randperm_path.exists():
    txt = randperm_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    perm = [int(x) for x in txt.replace('\n',' ').split() if re.match(r'^\d+$', x)]
    print('randperm 长度:', len(perm))
    # 构建逆排列 (索引 -> 原始位置)
    inverse = np.empty(len(perm), dtype=int)
    for i,p in enumerate(perm): inverse[p-1] = i  # 假设排列是 1-based
    print('逆排列示例(前10):', inverse[:10])
else:
    print('randperm 文件不存在')

### 解析 stations_list (站点数量与ID) 占位

In [ ]:
stations_path = DATA_DIR/'stations_list'
if stations_path.exists():
    lines = [l.strip() for l in stations_path.read_text(encoding='utf-8', errors='ignore').split('\n') if l.strip()]
    print('站点行数:', len(lines))
    print('前10站点ID:', lines[:10])
    if len(lines) == 963:
        print('站点数量与官方描述一致 (963).')
    else:
        print('站点数量与官方描述不一致, 需要核查.')
else:
    print('stations_list 文件不存在')

### 与官方描述核对总结 (预览基础)
- 行矩阵解析可行, 需全量运行确认所有天形状一致。
- 预览占有率范围是否严格在 [0,1]。若发现越界值需记录异常。
- 官方称无缺失值, 初步验证前10天无 NaN。
- 需进一步核对总天数 (官网 440 vs 文件 436?) 是否因为我们缺少额外假期/异常剔除说明。
- 后续可在资源允许情况下执行全量统计 (可能占用 ~300MB 内存, 建议流式处理聚合即可)。

### 基于实际文件的结构核对 (行数/站点列表)
使用系统命令统计得到: PEMS_train 行数=267, PEMS_test 行数=173。标签文件与站点列表均为单行。站点列表是一行包含所有 963 个 ID (而不是 963 行)。这与最初假设每行一个站点稍有不同。官网文档称 train 有 263 天, 这里出现 267 行, 需检查是否包含空行或额外异常日。下面代码重新精确统计非空行、解析每行矩阵形状。

In [ ]:
def count_nonempty_lines(path: pathlib.Path):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        return sum(1 for line in f if line.strip())
train_lines = count_nonempty_lines(train_path)
test_lines = count_nonempty_lines(test_path)
train_lines, test_lines

In [ ]:
# 逐行解析并统计成功的 963x144 矩阵数量与异常行索引
def audit_daily_file(path: pathlib.Path, expect_shape=(963,144), limit=None):
    ok = 0; bad = []; shapes = []
    for i, mat in enumerate(iter_day_matrices_optimized(path, limit=limit)):
        if mat.shape == expect_shape: ok += 1
        else: bad.append(i); shapes.append((i, mat.shape))
    return {'total_iterated': i+1 if 'i' in locals() else 0, 'ok_days': ok, 'bad_days': bad, 'bad_shapes': shapes}
audit_train_preview = audit_daily_file(train_path, limit=50)  # 仅前50天快速审计
audit_train_preview

### 站点列表单行解析
stations_list 为单行数字, 下面解析为数组并验证长度=963。

In [ ]:
if stations_path.exists():
    raw = stations_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    station_ids = [int(x) for x in raw.split() if re.match(r'^\d+$', x)]
    print('站点数量解析:', len(station_ids))
    print('前20个ID:', station_ids[:20])
else:
    print('stations_list 不存在')

### 差异与后续处理建议更新
1. 若最终确认有效训练天数应为 263, 需定位并剔除额外行(可能空行或损坏)。
2. 对所有行执行 audit_daily_file (不设 limit) 以确认 963x144 完整矩阵天数。
3. 标签文件是单行数组, 已与官网格式一致。
4. stations_list 单行格式解析成功, 后续可引入外部地理/道路元数据进行站点分组。

## 扩展分析: 满足导师要求汇总

### 1. 元数据表 (Meta Data Table)
包含: 文件, 天数(记录数), 传感器数, 时间段数, 总单元格数, 缺失值, 数值范围 (基于抽样/全量扫描的 min/max)。

In [ ]:
# 全量统计所需: 逐行解析训练/测试, 记录形状, 计算 min/max/缺失值 (占有率应无缺失)
train_total_lines = count_nonempty_lines(train_path)
test_total_lines  = count_nonempty_lines(test_path)
expected_shape = (963,144)
# 全局统计(流式): min/max/缺失/总值个数 + 抽样做分位数近似
train_stream_stats = stream_global_optimized(train_path, sample_target=1_000_000)
test_stream_stats  = stream_global_optimized(test_path,  sample_target=600_000)
train_stream_stats, test_stream_stats

### 2. 观测与特征数量 (Number of observations/features)
定义: 
- 观测(日级): 天数 (train/test)。
- 原子单元格级: 天数 * 传感器数 * 时间段数。
- 特征维度原始形式: (传感器ID, 时间槽) 二元。

In [ ]:
# 修正: 观测与特征数量 DataFrame 完整构造
sensors = expected_shape[0]; time_slots = expected_shape[1]
train_cells = train_total_lines * sensors * time_slots
test_cells  = test_total_lines  * sensors * time_slots
obs_feat_summary = pd.DataFrame([
    {'split':'train','days':train_total_lines,'sensors':sensors,'time_slots':time_slots,'total_cells':train_cells},
    {'split':'test','days':test_total_lines,'sensors':sensors,'time_slots':time_slots,'total_cells':test_cells},
    {'split':'total','days':train_total_lines+test_total_lines,'sensors':sensors,'time_slots':time_slots,'total_cells':train_cells+test_cells}
])
obs_feat_summary

### 3. 日级分布统计 (Per-day range & distribution)

In [ ]:
# 针对训练集构建日级统计 (min/max/mean/median/std) 与标签对应
def daily_stats(path: pathlib.Path, labels=None):
    rows = []
    for day_i, mat in enumerate(iter_day_matrices_optimized(path)):
        	vals = mat.ravel()
        	row = {
            'day_index': day_i,
            'min': float(vals.min()),
            'max': float(vals.max()),
            'mean': float(vals.mean()),
            'median': float(np.median(vals)),
            'std': float(vals.std())
        }
        	if labels is not None and day_i < len(labels): row['label'] = int(labels[day_i])
        	rows.append(row)
    return pd.DataFrame(rows)
daily_train_df = daily_stats(train_path, labels_train)
daily_test_df  = daily_stats(test_path,  labels_test)
daily_train_df.head()

In [ ]:
# 日级分布可视化: mean 与标签/星期几的关系 (箱线图)
plt.figure(figsize=(6,4))
sns.boxplot(data=daily_train_df, x='label', y='mean')
# 原：plt.title('Train: 日平均占有率按星期标签分布')
plt.title(L('train_daily_mean_boxplot'))
plt.xlabel('Label(weekdays)'); plt.ylabel('Daily Mean Occupancy'); plt.tight_layout(); plt.show()

### 4. 分组 (Special? In group?) 标签分组与平均日内曲线
对每个星期标签: 计算平均 24h 曲线 (对每天先对 963 传感器取均值 -> 144 长度向量, 再在组内求均值)。

In [ ]:
def label_mean_profiles(path: pathlib.Path, labels):
    # dict: label -> 累加向量 & 计数
    acc = {}
    for day_i, mat in enumerate(iter_day_matrices_optimized(path)):
        if day_i >= len(labels): break
        lb = int(labels[day_i])
        profile = mat.mean(axis=0)  # 传感器平均 -> 144
        if lb not in acc: acc[lb] = {'sum': np.zeros_like(profile), 'count':0}
        acc[lb]['sum'] += profile; acc[lb]['count'] += 1
    out = {}
    for lb, d in acc.items(): out[lb] = d['sum']/d['count']
    return out
profiles = label_mean_profiles(train_path, labels_train) if labels_train is not None else {}
# 绘制曲线
plt.figure(figsize=(10,5))
for lb in sorted(profiles.keys()):
    plt.plot(profiles[lb], label=f'Label {lb}')
# 原：plt.title('平均日内占有率曲线(按标签)')
plt.title(L('avg_daily_profile_by_label'))
# 原：plt.xlabel('时间槽(10分钟单位, 共144)')
plt.xlabel(L('time_slot_xlabel'))
# 原：plt.ylabel('平均占有率')
plt.ylabel(L('avg_occupancy_ylabel'))
plt.legend(ncol=2)
plt.tight_layout()
plt.show()
# 转为 DataFrame 展示前若干列
profiles_df = pd.DataFrame(profiles).T  # shape: labels x 144
profiles_df.head()

### 5. 缺失值 / 无效值 (Missing value / non-value)
验证占有率是否全部在 [0,1], 是否存在 NaN。

In [ ]:
def full_validity(path: pathlib.Path):
    nan_total=0; out_of_range=0; days=0
    for mat in iter_day_matrices_optimized(path):
        	days+=1
        	vals = mat.ravel()
        	nan_total += np.isnan(vals).sum()
        	out_of_range += np.sum((vals < 0) | (vals > 1))
    return {'days':days,'nan':int(nan_total),'out_of_range':int(out_of_range)}
valid_train = full_validity(train_path)
valid_test  = full_validity(test_path)
valid_train, valid_test

### 6. 特征列表 (First step: listing the feature)
原始数据单日矩阵维度=963(传感器) x 144(时间槽)。特征命名方案示例: occupancy_sensor_<stationId>_t<slot>。
下面生成少量示例特征名与站点ID对接。

In [ ]:
# 解析站点ID列表 (单行)
station_ids = station_ids if 'station_ids' in globals() else []
feature_name_examples = []
for sid in station_ids[:5]:
    for t in range(0,60,15):  # 示例: 每15个槽取一个
        feature_name_examples.append(f'occupancy_sensor_{sid}_t{t}')
feature_name_examples

### 7. 汇总元数据表 (结合以上统计)

In [ ]:
# 修正: meta_enhanced DataFrame 构造（防护式），若前面中断则重建所需统计
if 'train_stream_stats' not in globals():
    train_stream_stats = stream_global_optimized(train_path, sample_target=1_000_000)
if 'test_stream_stats' not in globals():
    test_stream_stats = stream_global_optimized(test_path, sample_target=600_000)
if 'valid_train' not in globals():
    valid_train = full_validity(train_path)
if 'valid_test' not in globals():
    valid_test = full_validity(test_path)
meta_enhanced = pd.DataFrame([
    {
        'split':'train',
        'days': train_total_lines,
        'sensors': sensors,
        'time_slots': time_slots,
        'total_cells': train_cells,
        'min': train_stream_stats.get('global_min', float('nan')),
        'max': train_stream_stats.get('global_max', float('nan')),
        'mean': train_stream_stats.get('mean', float('nan')),
        'std': train_stream_stats.get('std', float('nan')),
        'nan': train_stream_stats.get('nan_count', 0),
        'out_of_range': valid_train.get('out_of_range', 0) if isinstance(valid_train, dict) else 0
    },
    {
        'split':'test',
        'days': test_total_lines,
        'sensors': sensors,
        'time_slots': time_slots,
        'total_cells': test_cells,
        'min': test_stream_stats.get('global_min', float('nan')),
        'max': test_stream_stats.get('global_max', float('nan')),
        'mean': test_stream_stats.get('mean', float('nan')),
        'std': test_stream_stats.get('std', float('nan')),
        'nan': test_stream_stats.get('nan_count', 0),
        'out_of_range': valid_test.get('out_of_range', 0) if isinstance(valid_test, dict) else 0,
    }
])
meta_enhanced

### 8. 标签分布与类别不平衡指标

In [ ]:
if labels_train is not None:
    lab_counts = pd.Series(labels_train).value_counts().sort_index()
    imbalance_ratio = lab_counts.max()/lab_counts.min()
    display(lab_counts)
    print('最大/最小类样本比:', imbalance_ratio)

### 9. 全局近似分位数 (Train/Test)

In [ ]:
train_stream_stats['approx_quantiles'], test_stream_stats['approx_quantiles']

### 10. 要求实现完成总结
- Meta data 表: meta_enhanced 已给出。
- 观测与特征: obs_feat_summary + 单元格总量。
- Range & Distribution: min/max/分位数 + 日级箱线图。
- In group (Special?): 星期标签分组平均曲线 profiles。
- Missing value / non-value: valid_train/valid_test 验证无 NaN & 越界。
- Feature listing: feature_name_examples + 维度解释。
后续可扩展: 更多高级特征 (峰值时段, 早晚高峰差值等)。

In [ ]:
# 验证 train 天数与标签数量是否一致
train_labels_count = len(labels_train) if labels_train is not None else 0
match_status = (train_labels_count == train_total_lines)
{'train_days': train_total_lines, 'train_labels': train_labels_count, 'matched': match_status}

## 站点级高效可视化（面向 963 个站点）
目标：在不牺牲可读性的前提下，快速生成所有站点的箱线图与周内平均折线图。
改进要点：
- 流式读取 + 矩阵级聚合，避免逐图逐天重复计算。
- 先构建长表 DataFrame（station_id, label, daily_mean），一次性绘图或分页保存。
- 折线图：对每站点构建 7 条线（label=1..7），横轴 0..143 的时间槽。
- 箱线图：每站点一个小图（Box + Strip），支持分页保存至 Stage1-figure/。

In [ ]:
# 高效聚合：一次遍历训练集，得到两个长表：
# 1) station_label_daily_means: 每站点、每天的均值及其星期标签（用于箱线图）
# 2) station_weekday_time_profiles_long: 每站点、每星期标签、每时间槽的均值（用于折线图）
assert labels_train is not None, '训练标签未加载'
assert 'station_ids' in globals() and len(station_ids) == 963, 'stations_list 未解析或数量不为 963'
sensors = 963; time_slots = 144
# 预分配累加器： (station_index, label) -> sum(144), count
profiles_sum = np.zeros((sensors, 7, time_slots), dtype=float)
profiles_cnt = np.zeros((sensors, 7), dtype=int)
# 收集每日均值（用于箱线图）：list of dict
daily_rows = []
# 分批处理站点数据
batch_size = 500  # 每批处理 500 个站点
station_ids_batches = [station_ids[i:i + batch_size] for i in range(0, len(station_ids), batch_size)]
# 修复：在累加之前检查 mat 的形状是否与 profiles_sum 的形状匹配
# 如果不匹配，跳过该天的数据并记录异常
for day_i, mat in enumerate(iter_day_matrices_optimized(train_path)):
    if day_i >= len(labels_train):
        break
    lb = int(labels_train[day_i]) - 1  # 转为 0..6

    # 检查形状是否匹配
    if mat.shape[0] != sensors or mat.shape[1] != time_slots:
        print(f"警告: 第 {day_i} 天的矩阵形状 {mat.shape} 与预期 ({sensors}, {time_slots}) 不匹配，跳过。")
        continue

    # 站点日均值（144 槽均值）
    day_means = mat.mean(axis=1)  # shape: (963,)
    for batch in station_ids_batches:
        for ridx in range(len(batch)):
            daily_rows.append({
                'station_id': int(batch[ridx]),
                'label': lb + 1,
                'daily_mean': float(day_means[ridx])
            })

    # 周内时间槽均值累加
    profiles_sum[:, lb, :] += mat  # 对所有站点在该星期标签下累加 144 槽
    profiles_cnt[:, lb] += 1

station_label_daily_means = pd.DataFrame(daily_rows)

# 确保 station_weekday_time_profiles_long 定义正确
# 生成 station_weekday_time_profiles_long 数据框
station_weekday_time_profiles_long = []
for station_id in range(sensors):
    for label in range(7):
        if profiles_cnt[station_id, label] > 0:
            mean_profile = profiles_sum[station_id, label] / profiles_cnt[station_id, label]
            for slot, mean_occupancy in enumerate(mean_profile):
                station_weekday_time_profiles_long.append({
                    'station_id': station_ids[station_id],
                    'label': label + 1,
                    'slot': slot,
                    'mean_occupancy': mean_occupancy
                })
station_weekday_time_profiles_long = pd.DataFrame(station_weekday_time_profiles_long)

In [ ]:
# 分页保存箱线图（每页多子图），避免一次性绘制 963 个站点导致崩溃
def save_station_boxplots_paginated(df: pd.DataFrame, per_page=24, ncols=3, height_per_row=2.5, out_dir='Stage1-figure/box', dpi=150):
    import math
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
    station_ids_sorted = sorted(df['station_id'].unique().tolist())
    n_pages = math.ceil(len(station_ids_sorted) / per_page)
    for page in range(n_pages):
        start = page * per_page
        end = min(start + per_page, len(station_ids_sorted))
        subset_ids = station_ids_sorted[start:end]
        nrows = math.ceil(len(subset_ids) / ncols)
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 4, nrows * height_per_row))
        axes = axes.flatten()
        for ax, sid in zip(axes, subset_ids):
            sub = df[df['station_id'] == sid]
            sns.boxplot(data=sub, x='label', y='daily_mean', ax=ax, color='#88c')
            sns.stripplot(data=sub, x='label', y='daily_mean', ax=ax, color='#333', alpha=0.4, jitter=0.15)
            ax.set_title(L('station_daily_mean_title', sid=sid))
            ax.set_xlabel(L('station_plot_xlabel'))
            ax.set_ylabel(L('station_plot_ylabel'))
        for ax in axes[len(subset_ids):]: ax.axis('off')
        plt.tight_layout()
        save_path = pathlib.Path(out_dir) / f'page_{page+1}.png'
        plt.savefig(save_path, dpi=dpi)
        plt.close(fig)
# 执行保存（可调 per_page/ncols/dpi）
save_station_boxplots_paginated(station_label_daily_means, per_page=24, ncols=3, height_per_row=2.5, out_dir='Stage1-figure/box', dpi=150)
'Box plot pages saved to Stage1-figure/box'

In [ ]:
# 分页保存折线图：每站点一幅图，7 条线代表星期 1..7，横轴 0..143
def save_station_weekday_lines_paginated(df_long: pd.DataFrame, per_page=12, out_dir='Stage1-figure/line', dpi=150):
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
    station_ids_sorted = sorted(df_long['station_id'].unique().tolist())
    pages = [station_ids_sorted[i:i+per_page] for i in range(0, len(station_ids_sorted), per_page)]
    for page_i, subset_ids in enumerate(pages, start=1):
        n = len(subset_ids)
        fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(12, 3*n))
        if n == 1:
            axes = [axes]  # 如果只有一个子图，将 axes 转为列表以便统一处理
        for ax, sid in zip(axes, subset_ids):
            sub = df_long[df_long['station_id'] == sid]
            for lb in range(1, 8):
                s2 = sub[sub['label'] == lb].sort_values('slot')
                ax.plot(s2['slot'], s2['mean_occupancy'], label=f'{lb}')
            ax.set_title(L('station_weekday_profile_title', sid=sid))
            ax.set_xlabel(L('time_slot_xlabel'))
            ax.set_ylabel(L('avg_occupancy_ylabel'))
            ax.legend(title='Weekday', ncol=7, fontsize=8, title_fontsize=9)
        plt.tight_layout()
        save_path = pathlib.Path(out_dir) / f'page_{page_i}.png'
        plt.savefig(save_path, dpi=dpi)
        plt.close(fig)
# 执行保存（可调 per_page/dpi）
save_station_weekday_lines_paginated(station_weekday_time_profiles_long, per_page=12, out_dir='Stage1-figure/line', dpi=150)
'Line plot pages saved to Stage1-figure/line'